# Protocol 4 — Metabolic-State Crossover

Simulates Pred 4.A–4.C from Protocol 4 (*Metabolic-State Crossover: Allostatic Threshold Modulation Under
Energy Deficit*), updated to **protocol specification v1.1.0**. A four-session, two-site crossover compares
**fasted vs. fed** and **vigilance-depleted vs. rested** states. The two depletion modalities stay distinct.

All tests use α = 0.0167 (three-prediction family):

* **Pred 4.A** — State × Load interaction on d′ with p < 0.0167, BF₁₀ ≥ 100 and ηₚ² ≥ 0.06, **plus** the signed
  proportional margin Δd′_extero − Δd′_intero ≥ 0.15 × rested interoceptive d′ (Δ = depleted − rested; baseline must be positive).
* **Pred 4.B** — State × Load interaction on P3b (p < 0.0167): the interoceptive P3b reduction exceeds the
  exteroceptive reduction by ≥ 1.5 µV after arousal covariation.
* **Pred 4.C** — The selective interactions survive trial-level pupil and session RMSSD covariates.
  *Falsified*: covariates absorb the interaction (Δηₚ² ≥ 0.02, or Scenario E). Arousal adjustment alone does
  **not** establish metabolic-cost mediation.

> All data are synthetic. The θ shifts below are illustrative generative choices. The registered analysis is a
> probit GLMM (`seen ~ Signal × State × Load + Modality + Site + (1 + Signal × State | Participant)`); this
> notebook uses per-participant SDT d′ and paired contrasts as a transparent stand-in.
>
> **EP-4 | Paper 1 | Empirical | Depends on: EP-0 gate for secondary HEP analyses only |
> Spec v1.1.0, pre-collection | N enrolled = 64 (2 sites), analysable = 55, minimum = 48**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| N | 60 / 48 | 64 enrolled across 2 labs, 55 analysable (d = 0.45, α = 0.0167, power 0.80), 48 minimum |
| Sessions | Fed vs depleted (2) | 4: fasted, fed, vigilance-depleted, rested (≥ 48 h apart; 16 h fast; 600 kcal meal 2 h before) |
| Trials | 480 per cell | 320 per session: 3 contrasts (0.71/1.0/1.41 × c50) × 80 signal + 80 catch; 1280 total |
| α | 0.05 | 0.0167, with BF₁₀ ≥ 100 |
| Pred 4.A | ηₚ² only | ηₚ² + BF₁₀ + signed 15 % proportional d′ margin |
| Pred 4.C | BF₁₀ ≥ 100 | Survives pupil/RMSSD; falsified if Δηₚ² ≥ 0.02; adjustment ≠ mediation |
| Timescales | — | Deferred / exploratory; no framework vote |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.stats import norm, ttest_1samp
from scipy.stats import t as t_dist

from apgi.core import BETA_SM_DEFAULT, GAMMA_SIG_DEFAULT, compute_pi_i_eff, ignition_probability

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_4_metabolic_crossover.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 4 design (v1.1)

In [ ]:
P = PROTO["participants"]
BLOCKS, TRIALS = PROTO["blocks"], PROTO["trial_structure"]
N_ENROLLED, N_ANALYSABLE, N_SITES = P["n_target"], P["n_analyzable_target"], P["sites"]  # 64, 55, 2
SESSIONS = TRIALS["sessions"]                     # fasted, fed, vigilance_depleted, rested
MODALITIES = {"metabolic": ("fasted", "fed"), "vigilance": ("vigilance_depleted", "rested")}
LEVELS = np.array(BLOCKS["contrast_multipliers_c50"])  # 0.71, 1.0, 1.41
N_PER_LEVEL = 80
N_CATCH = int(BLOCKS["trials_per_block"] * BLOCKS["catch_fraction"])  # 80
assert len(LEVELS) * N_PER_LEVEL + N_CATCH == BLOCKS["trials_per_block"]
LOADS = ("intero", "extero")                      # interleaved interoceptive / exteroceptive streams
N_PER_LOAD = N_PER_LEVEL // 2

THR_A = PRED["Pred 4.A"]["confirming_thresholds"]
THR_B = PRED["Pred 4.B"]["confirming_thresholds"]
THR_C = PRED["Pred 4.C"]["confirming_thresholds"]
ALPHA = THR_A["alpha"]
KAPPA = PROTO["apgi_parameters"]["kappa"]["value"]  # None — uncalibrated

# Illustrative generative values (synthetic; not registered)
Z_CONTRAST = 1.6
# Per-modality depletion effects on θ: (interoceptive-selective, global fatigue, arousal level)
DEPLETION = {"fasted": (0.28, 0.05, 1.0), "vigilance_depleted": (0.20, 0.10, 1.5)}
AROUSAL_SHIFT = 0.06      # load-independent arousal effect on θ

print(f"{N_ENROLLED} enrolled across {N_SITES} sites; sessions {SESSIONS}")
print(f"{BLOCKS['trials_per_block']} trials/session ({N_PER_LEVEL} per contrast {LEVELS.tolist()} + {N_CATCH} catch); "
      f"{BLOCKS['total_experimental_trials']} total")
print(f"α = {ALPHA}, BF₁₀ ≥ {THR_A['bf10_minimum']}, ηₚ² ≥ {THR_A['eta_p_squared_minimum']}, "
      f"margin = {THR_A['proportional_margin']:.0%} of rested interoceptive d′")
rng = np.random.default_rng(2025)

## 2 — Simulate the crossover

In [ ]:
def simulate_session(session, theta0, pi_e, pi_i, sensitivity, rng):
    """Return d′, mean P3b, pupil and RMSSD for each load stream in one session."""
    d_intero, d_global, arousal_level = DEPLETION.get(session, (0.0, 0.0, 0.0))
    arousal = rng.normal(arousal_level, 0.4)
    out = {}
    for load in LOADS:
        theta = theta0 + d_global + AROUSAL_SHIFT * arousal + (sensitivity * d_intero if load == "intero" else 0.0)
        contrast = np.concatenate([np.repeat(LEVELS, N_PER_LOAD), np.zeros(N_CATCH // 2)])
        M_hat = rng.uniform(0.0, 0.5, len(contrast))
        gain = pi_i if load == "intero" else pi_e
        S = np.array([gain * Z_CONTRAST * c * abs(1 + 0.1 * rng.normal())
                      + 0.2 * compute_pi_i_eff(pi_i, BETA_SM_DEFAULT, float(m)) * abs(rng.normal())
                      for c, m in zip(contrast, M_hat)])
        p = np.array([ignition_probability(float(s), theta, GAMMA_SIG_DEFAULT) for s in S])
        seen = rng.random(len(S)) < p
        sig = contrast > 0
        hit, fa = seen[sig].mean(), seen[~sig].mean()
        dprime = norm.ppf((hit * sig.sum() + 0.5) / (sig.sum() + 1)) - norm.ppf((fa * (~sig).sum() + 0.5) / ((~sig).sum() + 1))
        p3b = np.mean(seen[sig] * (8.0 + 2.0 * (S[sig] - theta)) + rng.normal(0, 2.0, sig.sum()))
        out[load] = dict(dprime=dprime, p3b=p3b - 1.5 * AROUSAL_SHIFT * arousal)
    out["pupil"] = arousal + rng.normal(0, 0.3)
    out["rmssd"] = -arousal + rng.normal(0, 0.3)
    return out


site = np.repeat(np.arange(N_SITES), N_ENROLLED // N_SITES)
cohort = []
for pid in range(N_ENROLLED):
    pi_e, pi_i = rng.lognormal(0, 0.1), rng.lognormal(0, 0.1)
    theta0 = Z_CONTRAST * 1.0 + rng.normal(0, 0.05)  # QUEST-anchored c50
    sensitivity = max(0.0, rng.normal(1.0, 0.5))      # participant allostatic sensitivity
    cohort.append({s: simulate_session(s, theta0, pi_e, pi_i, sensitivity, rng) for s in SESSIONS})

analysable = np.sort(rng.choice(N_ENROLLED, N_ANALYSABLE, replace=False))  # attrition to the powered target
cohort = [cohort[i] for i in analysable]
n = len(cohort)
for s in SESSIONS:
    print(f"  {s:<19} d′ intero = {np.mean([c[s]['intero']['dprime'] for c in cohort]):.2f}   "
          f"d′ extero = {np.mean([c[s]['extero']['dprime'] for c in cohort]):.2f}")

## 3 — Prediction 4.A: State × Load on d′ with the signed margin

In [ ]:
def contrasts(measure, modality):
    dep, ctl = MODALITIES[modality]
    d_int = np.array([c[dep]["intero"][measure] - c[ctl]["intero"][measure] for c in cohort])
    d_ext = np.array([c[dep]["extero"][measure] - c[ctl]["extero"][measure] for c in cohort])
    base_int = np.array([c[ctl]["intero"][measure] for c in cohort])
    return d_int, d_ext, base_int


def interaction_stats(x):
    t, p = ttest_1samp(x, 0.0)
    return t, p, t**2 / (t**2 + len(x) - 1), jzs_bf10(t, len(x))


def arousal_adjusted(y, modality):
    """Intercept of y on centerd Δpupil and ΔRMSSD: the arousal-adjusted interaction."""
    dep, ctl = MODALITIES[modality]
    d_pupil = np.array([c[dep]["pupil"] - c[ctl]["pupil"] for c in cohort])
    d_rmssd = np.array([c[dep]["rmssd"] - c[ctl]["rmssd"] for c in cohort])
    X = np.column_stack([np.ones(n), d_pupil - d_pupil.mean(), d_rmssd - d_rmssd.mean()])
    beta, res, *_ = np.linalg.lstsq(X, y, rcond=None)
    df = n - X.shape[1]
    se0 = np.sqrt(res[0] / df * np.linalg.inv(X.T @ X)[0, 0])
    t_adj = beta[0] / se0
    return beta[0], t_adj, 2 * t_dist.sf(abs(t_adj), df), t_adj**2 / (t_adj**2 + df), df


results_A = {}
for mod in MODALITIES:
    d_int, d_ext, base_int = contrasts("dprime", mod)
    inter = d_ext - d_int                              # signed: positive = interoceptive hit harder
    t, p, eta, bf = interaction_stats(inter)
    margin = THR_A["proportional_margin"] * base_int.mean()
    ok_base = base_int.mean() > 0                      # non-positive baseline → secondary absolute rule only
    confirmed = p < ALPHA and bf >= THR_A["bf10_minimum"] and eta >= THR_A["eta_p_squared_minimum"] \
        and ok_base and inter.mean() >= margin
    falsified = p >= 0.05 and 1 / bf >= 6
    results_A[mod] = dict(inter=inter, t=t, p=p, eta=eta, bf=bf, margin=margin)
    print(f"{mod:<10} Δd′_ext − Δd′_int = {inter.mean():.3f} (margin {margin:.3f})  t({n - 1}) = {t:.2f}  "
          f"p = {p:.1e}  ηₚ² = {eta:.2f}  BF₁₀ = {bf:.3g}")
    print(f"           Pred 4.A ({mod}) → {verdict(confirmed, falsified)}")

## 4 — Prediction 4.B: P3b excess interoceptive reduction

In [ ]:
results_B = {}
for mod in MODALITIES:
    d_int, d_ext, _ = contrasts("p3b", mod)
    excess = d_ext - d_int                             # µV by which intero P3b drops more than extero
    t, p, eta, bf = interaction_stats(excess)
    adj_mean, _, p_adj, _, _ = arousal_adjusted(excess, mod)  # criterion applies after arousal covariation
    confirmed = p_adj < THR_B["alpha"] and adj_mean >= THR_B["excess_interoceptive_p3b_reduction_uv_minimum"]
    results_B[mod] = dict(excess=excess, p=p, eta=eta)
    print(f"{mod:<10} excess intero P3b reduction = {excess.mean():.2f} µV (arousal-adjusted {adj_mean:.2f} µV, "
          f"p = {p_adj:.1e})  → {verdict(confirmed)}")

## 5 — Prediction 4.C: Robustness to arousal covariates

Regress each participant's interaction contrast on the depletion-induced change in pupil diameter and RMSSD.
The intercept is the arousal-adjusted interaction. The interaction survives if it stays significant and
ηₚ² drops by less than 0.02.

In [ ]:
for mod in MODALITIES:
    for measure, y, eta_raw in [("d′", results_A[mod]["inter"], results_A[mod]["eta"]),
                                ("P3b", results_B[mod]["excess"], results_B[mod]["eta"])]:
        _, t_adj, p_adj, eta_adj, df = arousal_adjusted(y, mod)
        delta_eta = eta_raw - eta_adj
        confirmed = p_adj < THR_C["alpha"] and delta_eta < THR_C["eta_p_squared_reduction_strict_maximum"]
        falsified = delta_eta >= THR_C["eta_p_squared_reduction_strict_maximum"] or                     (p_adj >= 0.05 and 1 / jzs_bf10(t_adj, n) >= 6)
        print(f"{mod:<10} {measure:<3} adjusted t({df}) = {t_adj:.2f}, p = {p_adj:.1e}, "
              f"ηₚ² {eta_raw:.3f} → {eta_adj:.3f} (Δ = {delta_eta:+.3f})  → {verdict(confirmed, falsified)}")
print("Note:", PRED["Pred 4.C"]["claim"])

## 6 — Visualize Pred 4.A–4.C

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {"intero": "#2166ac", "extero": "#9966FF"}

ax = axes[0]
x = np.arange(len(SESSIONS))
for k, load in enumerate(LOADS):
    vals = np.array([[c[s][load]["dprime"] for s in SESSIONS] for c in cohort])
    ax.bar(x + (k - 0.5) * 0.38, vals.mean(0), yerr=vals.std(0) / np.sqrt(n), width=0.38,
           color=colors[load], alpha=0.85, edgecolor="white", capsize=3, label=load)
ax.set_xticks(x, ["Fasted", "Fed", "Vigilance\ndepleted", "Rested"], fontsize=8)
ax.set_ylabel("d′")
ax.set_title("Pred 4.A — d′ by session and load", fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
for k, mod in enumerate(MODALITIES):
    ax.hist(results_A[mod]["inter"], bins=14, alpha=0.6, label=f"{mod} (margin {results_A[mod]['margin']:.2f})",
            color=["#2166ac", "#00CC99"][k], edgecolor="white")
    ax.axvline(results_A[mod]["margin"], ls="--", color=["#2166ac", "#00CC99"][k], lw=1.2)
ax.axvline(0, color="k", lw=0.8, alpha=0.5)
ax.set_xlabel("Δd′_extero − Δd′_intero")
ax.set_ylabel("Participants")
ax.set_title("Pred 4.A — signed interaction vs 15 % margin", fontsize=10)
ax.legend(fontsize=7)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
means = [results_B[m]["excess"].mean() for m in MODALITIES]
sems = [results_B[m]["excess"].std() / np.sqrt(n) for m in MODALITIES]
ax.bar(list(MODALITIES), means, yerr=sems, color=["#2166ac", "#00CC99"], alpha=0.85, edgecolor="white",
       width=0.45, capsize=5)
ax.axhline(THR_B["excess_interoceptive_p3b_reduction_uv_minimum"], ls="--", lw=1, color="k", label="1.5 µV criterion")
ax.set_ylabel("Excess intero P3b reduction (µV)")
ax.set_title("Pred 4.B — P3b selectivity", fontsize=10)
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Protocol 4 — Metabolic-state crossover · spec v1.1.0 · synthetic N = {n} (2 sites)", fontsize=12)
plt.tight_layout()
plt.show()